# 02 · Map the BIODEP gene panel to microarray probes

**Aim:** find which probes on the Affymetrix U133 Plus 2.0 array measure the 16 genes of the BIODEP panel ([Cattaneo et al., 2020](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7376244/)).

In [1]:
import warnings
from pathlib import Path

import GEOparse
import pandas as pd

raw_dir = Path("../data/raw")
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=pd.errors.DtypeWarning)
    gse = GEOparse.get_GEO(geo="GSE98793", destdir=str(raw_dir), silent=True)

platform = gse.gpls["GPL570"].table
print(platform.shape)
platform.columns.tolist()

(54675, 16)


['ID',
 'GB_ACC',
 'SPOT_ID',
 'Species Scientific Name',
 'Annotation Date',
 'Sequence Type',
 'Sequence Source',
 'Target Description',
 'Representative Public ID',
 'Gene Title',
 'Gene Symbol',
 'ENTREZ_GENE_ID',
 'RefSeq Transcript ID',
 'Gene Ontology Biological Process',
 'Gene Ontology Cellular Component',
 'Gene Ontology Molecular Function']

## 1. The BIODEP gene panel
The 16 genes measured by qPCR in the BIODEP study, grouped by biological family. GR is listed under its official symbol, NR3C1. This list is fixed before looking at expression values.

In [2]:
panel = {
    "IL1B": "inflammation",
    "IL6": "inflammation",
    "TNF": "inflammation",
    "MIF": "inflammation",
    "CCL2": "inflammation",
    "CXCL12": "inflammation",
    "CRP": "inflammation",
    "A2M": "inflammation",
    "P2RX7": "inflammation",
    "NR3C1": "glucocorticoid",
    "FKBP5": "glucocorticoid",
    "SGK1": "glucocorticoid",
    "ISG15": "interferon",
    "STAT1": "interferon",
    "USP18": "interferon",
    "AQP4": "other",
}

len(panel)

16

In [3]:
matches = []

for probe_id, symbol_text in platform[["ID", "Gene Symbol"]].dropna().values:
    genes_on_probe = symbol_text.split(" /// ")
    for gene in genes_on_probe:
        if gene in panel:
            matches.append({
                "probe": probe_id,
                "gene": gene,
                "family": panel[gene],
                "n_genes_on_probe": len(genes_on_probe),
            })

probe_map = pd.DataFrame(matches)
probe_map["gene"].value_counts()

gene
STAT1     6
NR3C1     5
AQP4      5
FKBP5     3
A2M       2
CXCL12    2
IL1B      2
CRP       2
P2RX7     2
SGK1      1
IL6       1
ISG15     1
TNF       1
CCL2      1
MIF       1
USP18     1
Name: count, dtype: int64

In [4]:
probe_map["suffix"] = probe_map["probe"].str.extract(r"(_s_at|_x_at|_at)$")
probe_map.sort_values(["family", "gene"])

,probe,gene,family,n_genes_on_probe,suffix
6,204560_at,FKBP5,glucocorticoid,1,_at
25,224840_at,FKBP5,glucocorticoid,1,_at
26,224856_at,FKBP5,glucocorticoid,1,_at
3,201865_x_at,NR3C1,glucocorticoid,1,_x_at
4,201866_s_at,NR3C1,glucocorticoid,1,_s_at
19,211671_s_at,NR3C1,glucocorticoid,1,_s_at
20,216321_s_at,NR3C1,glucocorticoid,1,_s_at
29,232431_at,NR3C1,glucocorticoid,1,_at
2,201739_at,SGK1,glucocorticoid,1,_at
0,1558450_at,A2M,inflammation,1,_at


## 3. Expression data
Load the expression values (one row per probe, one column per sample) and check how they were processed.

In [5]:
pheno = gse.phenotype_data
print(pheno["data_processing"].unique())

expr = gse.pivot_samples("VALUE")
print(expr.shape)
expr.iloc[:, :5].describe().round(2)

<StringArray>
['The data were read and RMA-normalized using the Bioconductor package affy (default parameters).']
Length: 1, dtype: str
(54675, 192)


name,GSM2612096,GSM2612097,GSM2612098,GSM2612099,GSM2612100
count,54675.00,54675.00,54675.00,54675.00,54675.00
mean,5.44,5.42,5.39,5.38,5.43
std,2.31,2.36,2.45,2.40,2.34
min,1.43,1.38,1.34,1.41,1.50
25%,3.63,3.55,3.40,3.45,3.60
50%,5.04,4.94,4.93,4.89,5.00
75%,6.89,6.99,7.10,6.99,6.94
max,14.19,14.20,14.25,14.25,14.22


In [6]:
pheno = gse.phenotype_data
print(pheno["data_processing"].unique())

expr = gse.pivot_samples("VALUE")
print(expr.shape)
expr.iloc[:, :10].describe().round(2)

<StringArray>
['The data were read and RMA-normalized using the Bioconductor package affy (default parameters).']
Length: 1, dtype: str
(54675, 192)


name,GSM2612096,GSM2612097,GSM2612098,GSM2612099,GSM2612100,GSM2612101,GSM2612102,GSM2612103,GSM2612104,GSM2612105
count,54675.00,54675.00,54675.00,54675.00,54675.00,54675.00,54675.00,54675.00,54675.00,54675.00
mean,5.44,5.42,5.39,5.38,5.43,5.42,5.43,5.41,5.39,5.41
std,2.31,2.36,2.45,2.40,2.34,2.38,2.38,2.36,2.38,2.37
min,1.43,1.38,1.34,1.41,1.50,1.51,1.32,1.47,1.42,1.47
25%,3.63,3.55,3.40,3.45,3.60,3.52,3.52,3.54,3.49,3.51
50%,5.04,4.94,4.93,4.89,5.00,4.89,4.92,4.99,4.95,4.96
75%,6.89,6.99,7.10,6.99,6.94,7.04,7.04,6.96,6.97,6.97
max,14.19,14.20,14.25,14.25,14.22,14.14,14.11,14.29,14.28,14.27
